#### TextCNN
- CNN(합성곱 신경망) : 이미지 분석시 사용하는 신경망 모델 
- 이미지에서 사용이 되는 CNN을 이용하여 텍스트를 이미지와 같다 라고 가정을 하고 설계한 모델 
1. 문장의 행렬화 (임베딩)
    - 배치로 문장을 모은다. ( 배치사이즈, 문장의 길이(토큰 개수) )
    - 임베딩 처리를 통해서 ( 배치사이즈, 문장의 길이, 임베딩 차원의 수 )
2. 텍스트 돋보기로 문맥을 읽기 
    - 단어의 흐름 방향으로 구간은 선택하여 데이터 학습 
3. 맥스풀링(가장 큰 값을 찾는 과정) 
    - 2번 과정에서의 구간 데이터들을 합성곱을 통해 가장 큰 값을 선택하는 과정 
    - 가장 강한 인상 남기기
4. 최종 분류 
    - 해당 데이터셋을 이용하여 최종 분류 하는 과정 
    - kim CNN 구조는 단어를 3, 4, 5로 분류하여 맥스풀링을 한 뒤 분류 
    - 과적합의 위험성 때문에 dropout()을 이용하여 일정 피쳐를 0으로 만들어서 과적합 방지 

#### TextCNN 사용하기 전 데이터를 준비 
1. 데이터 로드 
2. 데이터 튜닝 
3. 데이터 토큰화 
4. 단어 사전 등록
5. 단어 사전을 이용한 인코딩 

In [1]:
import pandas as pd 
import re 
from konlpy.tag import Komoran
from collections import Counter

In [2]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [3]:
df.dropna(inplace=True)

In [4]:
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()
    return text

In [5]:
df['document'] = df['document'].map(normalize)

In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149995 non-null  int64
 1   document  149995 non-null  str  
 2   label     149995 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [7]:
# 빈 텍스트 제외 
df = df.loc[
    ~(df['document'] == ''), 
]

In [8]:
# 중복 데이터 제거 
df.drop_duplicates('document', inplace=True)

In [9]:
komoran = Komoran()

allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR']
stop_word = ['하다', '되다', '이다']

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if word not in stop_word and pos in allow_pos:
            tokens.append(word)
    return tokens

In [10]:
df2 = df[:1000]

In [ ]:
texts, labels = df2['document'].values, df2['label'].values

# texts 토큰화 
tokens_list = [tokenize(text) for text in texts]
tokens_list

In [ ]:
# 최소 등장 횟수에 따른 단어 사전을 생성 
Counter(word for toks in tokens_list for word in toks)

In [14]:
t_s = []
for toks in tokens_list:
    for word in toks:
        t_s.append(word)
freq = Counter(t_s)

In [15]:
freq.items()

dict_items([('더빙', 5), ('진짜', 68), ('짜증', 16), ('나', 26), ('목소리', 2), ('포스터', 8), ('초딩', 3), ('영화', 364), ('오버', 3), ('연기', 58), ('가볍', 3), ('교도소', 1), ('이야기', 14), ('솔직히', 14), ('재미', 31), ('없', 114), ('평점', 27), ('조정', 1), ('익살', 1), ('돋보이', 3), ('스파이더맨', 1), ('늙', 3), ('보이', 20), ('하', 133), ('커스틴 던스트', 1), ('너무나', 7), ('막', 8), ('걸음마', 1), ('떼', 3), ('초등학교', 4), ('학년', 3), ('용', 2), ('별', 2), ('반개', 3), ('아깝', 31), ('원작', 6), ('긴장감', 6), ('제대로', 2), ('살리', 5), ('욕', 7), ('나오', 52), ('이응경', 1), ('길용우', 1), ('생활', 2), ('이', 26), ('정말', 67), ('발로', 2), ('납치', 1), ('감금', 1), ('반복', 3), ('드라마', 28), ('가족', 6), ('못하', 3), ('사람', 44), ('모이', 1), ('액션', 18), ('있', 86), ('안', 66), ('왜', 43), ('낮', 8), ('꽤', 8), ('보', 262), ('헐리우드', 2), ('화려', 5), ('너무', 61), ('길들이', 1), ('볼', 12), ('때', 34), ('눈물', 6), ('나서', 6), ('죽', 14), ('향수', 2), ('자극', 5), ('허진호', 1), ('감성', 3), ('절제', 3), ('멜로', 6), ('달인', 1), ('울', 9), ('손들', 1), ('횡단보도', 1), ('건너', 1), ('뛰쳐나오', 1), ('이범수', 1), ('드럽', 4), ('담백', 2),

In [16]:
# 최소 등장 횟수는 2회 
min_count = 2

# 단어 사전에 특수 토큰 <PAD> <UNK> 토큰을 먼저 대입 
vocab = ['<PAD>', '<UNK>']

for word, cnt in freq.items():
    # word : 단어
    # cnt : 출현 횟수
    if cnt >= min_count:
        vocab.append(word)

In [17]:
len(vocab)

1046

In [ ]:
stoi = {word : idx for idx, word in enumerate(vocab)}
stoi

In [ ]:
stoi2 = dict()
for idx, word in enumerate(vocab):
    # idx : 위치 값
    # word : 단어 
    stoi2[word] = idx

stoi2